# Internal coordinates and averages

This notebook focuses on internal coordinates: distances, angles and dihedral angles.
The name "internal coordinates" refers to the fact that they are insensitive to global rotations or translations of the system.

These internal coordinates will also be used to demonstrate the computation of averages of time series and the error on such averages.

The analysis uses the output files `init.pdb`, `scalars.csv` and `traj.dcd` of the villin headpiece simulation in [../04_protein/01_villin_headpiece.ipynb](../04_protein/01_villin_headpiece.ipynb).
These files must first be copied to this directory, as explained at the end of that notebook.

In [ ]:
%matplotlib widget
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import mdtraj
import nglview
import numpy as np
import pandas
from stacie import PadeModel, UnitConfig, compute_spectrum, estimate_acint, plot_fitted_spectrum

# Local import, see block_averaging.py in the same directory.
from block_averaging import averror, num_independent

The following cell loads the trajectory and the scalar properties, and visualizes the protein with NGLView.
We will often need atom indices in this notebook.
After clicking on an atom in the viewer, the details of the picked atom can be shown by running the code cell below the viewer.

In [ ]:
traj = mdtraj.load("traj.dcd", top="init.pdb")
df = pandas.read_csv("scalars.csv")
view = nglview.show_mdtraj(traj)
view.add_representation("ball+stick", selection="protein")
view

In [ ]:
# Rerun this cell after clicking on an atom in the viewer above.
# The atom index is listed as "index" under "atom1".
view.picked

Atom indices can also be looked up with the [atom selection language](https://www.mdtraj.org/development/atom_selection.html) of MDTraj, which is more reproducible than clicking.
For example, the following cell prints the indices of the C-S-C atoms in the side chain of methionine 12.
(`resSeq` is the residue number in the PDB file.)

In [ ]:
print(traj.topology.select("protein and resSeq 12 and name CG SD CE"))

## 1. Inter-atomic distances

The following example computes two bond lengths as a function of time, namely for the `N-CA` and `CA-C` bonds in the first residue (leucine).
Note that all indices in Python start counting from zero, so the bond between the first two atoms in the PDB and DCD files is denoted by `[0, 1]`.

The documentation of `compute_distances` can be found here: https://www.mdtraj.org/development/api/generated/mdtraj.compute_distances.html

In [ ]:
distances = mdtraj.compute_distances(traj, [[0, 1], [1, 2]])
labels = ["N-CA (LEU1)", "CA-C (LEU1)"]
print(distances.shape)
# The first distance in frame 400
print(distances[400, 0])
plt.close("dist")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="dist")
for label, col in zip(labels, distances.T, strict=True):
    ax.plot(df["Time (ps)"], col, ".", label=label)
ax.set_xlabel("Time [ps]")
ax.set_ylabel("Distances [nm]")
ax.legend(loc=0)

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> For the simple bond lengths shown above, the initialization (warming up of the protein during the first 5 picoseconds) seems to have little effect.
> Find the atom indices of a few strongly electrostatically interacting atom pairs that are not covalently bonded,
> e.g. the nitrogen in a lysine side chain interacting with the carboxylate group of an aspartate or glutamate side chain, or of the C-terminus.
> Plot the distances between two pairs of electrostatically interacting atoms.
> Can you recognize the thermalization in these inter-atomic distances?
> Add two more distances, one for a hydrogen bond in an alpha helix and one outside an alpha helix.

## 2. Valence and dihedral angles

A similar analysis for the C-S-C angle in the side chain of MET12, the backbone dihedral angle $\psi$ of LEU1 and the backbone dihedral angle $\omega$ of the LEU1-SER2 peptide bond
is carried out with [compute_angles](https://www.mdtraj.org/development/api/generated/mdtraj.compute_angles.html) and [compute_dihedrals](https://www.mdtraj.org/development/api/generated/mdtraj.compute_dihedrals.html) in the following code cell.

In [ ]:
angles = mdtraj.compute_angles(traj, [[171, 172, 173]])
dihedrals = mdtraj.compute_dihedrals(traj, [[0, 1, 2, 21], [1, 2, 21, 22]])
print(angles.shape)
print(dihedrals.shape)
plt.close("angle")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="angle")
ax.plot(df["Time (ps)"], np.degrees(angles[:, 0]), ".", label="C-S-C angle (MET12)")
ax.plot(
    df["Time (ps)"],
    np.degrees(dihedrals[:, 0]),
    ".",
    label="N-CA-C-N dihedral (psi LEU1)",
)
ax.plot(
    df["Time (ps)"],
    np.degrees(dihedrals[:, 1]),
    ".",
    label="CA-C-N-CA dihedral (omega LEU1-SER2)",
)
ax.set_xlabel("Time [ps]")
ax.set_ylabel("Angles [deg]")
ax.legend(loc=0)

Valence angles in proteins are usually rather stiff, while the dihedral angles in the backbone ($\phi$ and $\psi$) and in the side chains ($\chi_n$) explain most of the conformational changes.
The backbone dihedral angle $\omega$ is an exception: due to the partial double-bond character of the peptide bond, it remains close to $180^\circ$ (trans).
Because $180^\circ$ and $-180^\circ$ represent the same dihedral angle, the values of $\omega$ jump between the top and the bottom of the plot.
We will return to this issue in section 3.

## 3. Average of a dihedral angle

The average of a dihedral angle can become meaningless because these angles are periodic, i.e. $\phi=35^\circ$ and $\phi=395^\circ$ represent the same dihedral angle.
As a consequence, dihedral angles exceeding $180^\circ$ will continue at $-180^\circ$.
Such discontinuous jumps result in meaningless averages.
The $\omega$ dihedral angle of a peptide bond is an extreme example:
it fluctuates around $180^\circ$, so its values are split over both ends of the interval $[-180^\circ, 180^\circ]$.

This problem can be surmounted by computing a slightly different type of average, i.e. the average cosine and sine of the angle, followed by a conversion back into an angle.
For example:

In [ ]:
x = np.cos(dihedrals[:, 1]).mean()
y = np.sin(dihedrals[:, 1]).mean()
avdihed1 = np.arctan2(y, x)
avdihed1_wrong = dihedrals[:, 1].mean()
print("Average of cosine and sine [deg]:", np.degrees(avdihed1))
print("Naive average [deg]:", np.degrees(avdihed1_wrong))
plt.close("dihed1")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="dihed1")
ax.plot(
    df["Time (ps)"],
    np.degrees(dihedrals[:, 1]),
    ".",
    label="CA-C-N-CA dihedral (omega LEU1-SER2)",
)
ax.axhline(np.degrees(avdihed1), color="k", label="Average of cosine and sine")
ax.axhline(np.degrees(avdihed1_wrong), color="r", label="Naive average")
ax.set_xlabel("Time [ps]")
ax.set_ylabel("Angles [deg]")
ax.legend(loc=0)

The naive average lies far from $\pm 180^\circ$, because it combines values close to $180^\circ$ with values close to $-180^\circ$, which partially cancel each other.
The alternative approach does not suffer from this issue.
To understand why this works, consider a plot of the sine versus the cosine:

In [ ]:
plt.close("dihed2")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="dihed2")
ax.plot(np.cos(dihedrals[:, 1]), np.sin(dihedrals[:, 1]), "k+", alpha=0.1)
ax.plot([x], [y], "ro")
ax.plot([0, x], [0, y], "r-")
# Matplotlib draws arcs counterclockwise from theta1 to theta2.
avdihed1_deg = np.degrees(avdihed1)
ax.add_patch(
    mpatches.Arc(
        [0, 0],
        1,
        1,
        angle=0,
        theta1=min(0, avdihed1_deg),
        theta2=max(0, avdihed1_deg),
        color="r",
    )
)
ax.axhline(0, color="k")
ax.axvline(0, color="k")
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect("equal")

When plotting the dihedral angles as points on a circle, i.e. $x=\cos\phi$ and $y=\sin\phi$, there is no longer a discontinuity and the average is well-behaved.
The angle is then derived from the average (red dot) by computing the angle with the X-axis, using the `arctan2` function.
A few remarks:

- The function `arctan2` takes the Y-coordinate as first argument.

- This approach will not work when the dihedral angles are uniformly distributed over the interval $[-180^\circ,180^\circ]$.
  In this case, the average cosine and sine are (nearly) zero and the angle with the X-axis is not (or ill) defined.

## 4. Histogram and free energy

Histograms may also be convenient to characterize an internal coordinate.
We will apply it here to the C-S-C angle in the side chain of MET12, which is a relatively stiff mode.
The angles are almost perfectly [normally distributed](https://en.wikipedia.org/wiki/Normal_distribution), which can be shown by overlaying the histogram with the normal probability density.

For reference, the force-field parameters for this angle can be found here:

https://github.com/openmm/openmm/blob/8.6.1/wrappers/python/openmm/app/data/amber14/protein.ff14SB.xml#L3077

and read

```
<Angle angle="1.726130630222392" k="518.816" type1="protein-CT" type2="protein-S" type3="protein-CT"/>
```

The equilibrium angle is $1.726130630222392 \text{ rad}$, which corresponds to $98.9^\circ$.
The force constant $k$ is expressed in $\mathrm{kJ}\,\mathrm{mol}^{-1}\,\mathrm{rad}^{-2}$, and the energy of this force-field term is $\frac{1}{2}k(\theta-\theta_0)^2$.

In [ ]:
plt.close("histangle")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="histangle")
bins = np.radians(np.arange(85, 115, 1))
ava = angles[:, 0].mean()
avs = angles[:, 0].std()
theta = np.radians(np.linspace(80, 120, 79))
# The following equation is the probability density of a
# univariate normal distribution.
density = np.exp(-((theta - ava) ** 2) / (2 * avs**2)) / np.sqrt(2 * avs**2 * np.pi)
# Angles are converted to degrees for plotting.
# The histogram of angles in degrees is a probability density in 1/deg,
# so the normal density must also be converted from 1/rad to 1/deg.
ax.hist(np.degrees(angles[:, 0]), np.degrees(bins), density=True)
ax.plot(np.degrees(theta), density * np.pi / 180)
ax.axvline(np.degrees(ava), color="k")
ax.set_title(f"Average = {np.degrees(ava):.1f} deg, Std.Dev. = {np.degrees(avs):.1f} deg")
ax.set_xlabel("C-S-C angle [deg]")
ax.set_ylabel("Probability density [1/deg]")

The average appears to differ slightly from the equilibrium value, which may be due to other force-field terms influencing the local geometry.
Whether this difference is significant is discussed at the end of section 6, after the error on the average has been introduced.

The empirical probabilities from the histogram can be translated into free energies (up to a constant) by employing the following relationship:

$$F(\theta) = -k_B T \ln(p(\theta))$$

When this relation is applied to the normal probability density, one obtains a quadratic approximation of the free energy, with a force constant $k_B T/\sigma^2$.

In [ ]:
boltzmann = 1e-3 * 1.380649e-23 * 6.02214076e23  # in kJ/(mol K)
temperature = 300
# Force field parameters
theta0_par = 1.726130630222392  # rest angle in rad
fc_par = 518.816  # force constant in kJ/mol/rad^2
f0 = 0.5 * fc_par * (theta - theta0_par) ** 2
# Quadratic approximation
fc = boltzmann * temperature / avs**2  # force constant in kJ/mol/rad^2
f = 0.5 * fc * (theta - ava) ** 2
# Empirical model, using histogram
pe = np.histogram(angles[:, 0], bins, density=True)[0]
with np.errstate(divide="ignore"):
    fe = -boltzmann * temperature * np.log(pe)
fe -= fe.min()

plt.close("free")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="free")
# Angles are converted to degrees for plotting.
ax.plot(np.degrees(theta), f0, label="Force field valence angle term")
ax.plot(np.degrees(theta), f, label="Quadratic free energy model")
ax.plot(np.degrees((bins[1:] + bins[:-1]) / 2), fe, label="Empirical free energy")
ax.set_xlabel("C-S-C angle [deg]")
ax.set_ylabel("Free energy [kJ/mol]")
ax.legend(loc=0)

# Finally, the force constant is printed for comparison with the force-field parameter.
print("Force constant [kJ/mol/rad^2]:", fc)

Also here, the free-energy force constant appears to deviate from the AMBER force-field parameter.
Again, this could be due to an influence from the environment of the C-S-C angle.

## 5. Averages, fluctuations and error on the average

Computing averages and fluctuations (standard deviations) of a time-dependent data series is easily carried out with NumPy.
The following code cell contains some examples.

In [ ]:
print("average distance 0 [nm]", distances[:, 0].mean())
print("average distance 0 [nm]", np.mean(distances[:, 0]))
print("all average distances [nm]", distances.mean(axis=0))
print("all average distances [nm]", np.mean(distances, axis=0))
print("st.dev. distance 0 [nm]", distances[:, 0].std())
print("st.dev. distance 0 [nm]", np.std(distances[:, 0]))
print("all st.dev. distances [nm]", distances.std(axis=0))
print("all st.dev. distances [nm]", np.std(distances, axis=0))

Properties computed from an MD simulation behave like random variables.
Strictly speaking, they are deterministic yet chaotic and are often driven by pseudorandom numbers in the initial velocities and thermostat.
Consequently, the average over a finite number of steps is also a random variable:
it varies from one simulation to the next and differs from the exact ensemble average.
The goal is to quantify this statistical uncertainty.
This is not trivial because consecutive time steps are correlated, meaning that the effective number of independent samples is much smaller than the number of steps.

When computing an average over $N$ **uncorrelated** data points, the error on the average is

$$
\sigma_{\langle a \rangle} = \frac{\sigma_a}{\sqrt{N-1}}
\qquad \text{with} \qquad
\sigma_a = \sqrt{\frac{1}{N}\sum_{i=1}^N (a_i - \langle a \rangle)^2}
$$

where $\sigma_a$ is the standard deviation, as computed with `np.std` above.

This formula is rarely useful for computing uncertainties on averages (over time) of quantities from MD simulations, because the values for subsequent time steps tend to be correlated.
(See e.g. the inter-atomic distances for electrostatically interacting pairs of atoms.)

To account for time correlations in the error estimate, one should then replace $N$ in the denominator by the number of uncorrelated data points in the time series.
The function `num_independent`, imported from `block_averaging.py`, estimates this number of independent samples with the block-averaging method.
Its implementation is based on the description of the block-averaging method in the book "Computer Simulation of Liquids" by M.P. Allen and D.J. Tildesley (second edition, 2017), section 8.4.1 on page 282.
With the `fignum` argument, it also plots the statistical inefficiency as a function of the block size, together with the model used to extrapolate it to an infinite block size.
The quality indicator should be at least 5, otherwise the time series is too short for a reliable estimate.
The function `averror` then computes the error on the average with the formula above.

In [ ]:
values = distances[:, 0]
# values = np.degrees(angles[:, 0])
# values = np.degrees(dihedrals[:, 0])
num_indep, quality = num_independent(values, fignum="numindep")
print("Quality:", quality)
print("Number of values:", len(values))
print("Number of independent values:", num_indep)
print("Error on the average:", averror(values, num_indep))

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Try estimating the error on the average of all internal coordinates computed so far. Does the number of independent samples correlate with the time dependence plotted previously?

## 6. Error on the average with STACIE

Block averaging is easy to understand, but its implementation here is (too) simple.
A more robust alternative is [STACIE](https://molmod.github.io/stacie/), a Python package that estimates the integral of the autocorrelation function (ACF) of a time series from its power spectrum.
The variance of the average of a time series is related to this integral:

$$
\mathrm{var}[\langle a \rangle] \approx \frac{1}{t_\text{sim}} \int_{-\infty}^{+\infty} c(\Delta_t)\,\mathrm{d}\Delta_t
$$

where $c(\Delta_t)$ is the autocovariance function and $t_\text{sim}$ the total length of the simulation.
For a given time series, STACIE computes the power spectrum, scaled such that its zero-frequency limit is this variance,
and fits a model to the low-frequency part of the spectrum.
(The fit uses the Padé model, the default recommendation of the STACIE documentation for error estimates.)
Unlike block averaging, STACIE also provides an uncertainty on the estimate of the error on the average,
which tells you when the estimate cannot be trusted.
The details are explained in the [STACIE documentation](https://molmod.github.io/stacie/stable/properties/error_estimates.html).

The following cell defines a function `stacie_error` that returns the error on the average, the uncertainty on this error, and the full STACIE result.
The number of independent samples is derived from the error on the average,
by comparing it to the standard deviation of the time series: $N_\text{indep} = \sigma_a^2 / \sigma_{\langle a \rangle}^2$.

In [ ]:
timestep = df["Time (ps)"][1] - df["Time (ps)"][0]


def stacie_error(values, timestep):
    """Estimate the error on the average of a time-correlated series with STACIE."""
    spectrum = compute_spectrum(
        values,
        # The factor 2 is needed for the variance of the average.
        prefactors=2.0 / (timestep * len(values)),
        timestep=timestep,
        # The zero-frequency component contains the average, which must be excluded.
        include_zero_freq=False,
    )
    result = estimate_acint(spectrum, PadeModel([0, 2], [2]))
    error = np.sqrt(result.acint)
    # Propagation of the uncertainty on the variance to the error.
    error_std = 0.5 * result.acint_std / error
    return error, error_std, result


values = distances[:, 0]
error, error_std, result = stacie_error(values, timestep)
print(f"Error on the average: {error:.2e} +- {error_std:.1e}")
print("Number of independent values:", np.var(values) / error**2)
print("Statistical inefficiency: ")
print("Integrated autocorrelation time [ps]:", result.corrtime_int)

plt.close("stacie")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="stacie")
uc = UnitConfig(
    time_unit_str="ps",
    time_fmt=".2f",
    freq_unit_str="1/ps",
    acint_unit_str="nm$^2$",
    acint_fmt=".1e",
)
plot_fitted_spectrum(ax, uc, result)

The plot shows the power spectrum (blue dots) with the fitted model (green).
The zero-frequency limit of the model is the variance of the average.
The red dotted curve shows which part of the spectrum contributes to the fit:
STACIE automatically selects the frequency range in which the model describes the spectrum well.
The fit is only meaningful if enough spectrum points contribute to it.

The following cell lists the average of each internal coordinate (with angles converted to degrees)
and the errors on the average obtained with the naive formula, with block averaging and with STACIE.
For the bond length, whose ACF decays quickly, both approaches give errors of the same order of magnitude (they differ by about 20%).
When the time series is too short for a reliable estimate of the error, as for the dihedral angle,
the uncertainty reported by STACIE is huge.

In [ ]:
print(
    f"{'':15s} {'unit':>4s} {'average':>9s} {'naive':>9s} {'block averaging':>16s} {'STACIE':>20s}"
)
# The scale factor converts angles from radians to degrees before printing.
for label, unit, scale, values in [
    ("N-CA distance", "nm", 1.0, distances[:, 0]),
    ("C-S-C angle", "deg", 180 / np.pi, angles[:, 0]),
    ("psi dihedral", "deg", 180 / np.pi, dihedrals[:, 0]),
]:
    error_naive = averror(values)
    num_indep_block, quality = num_independent(values)
    error_block = averror(values, num_indep_block)
    error_stacie, error_stacie_std, _ = stacie_error(values, timestep)
    print(
        f"{label:15s} {unit:>4s} {scale * values.mean():9.5f} {scale * error_naive:9.2e} "
        f"{scale * error_block:16.2e} "
        f"{scale * error_stacie:12.2e} +- {scale * error_stacie_std:7.1e}"
    )

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Use the table above to interpret the C-S-C angle of MET12 from section 4.
> The force-field equilibrium angle is $98.9^\circ$ (see section 4).
>
> - How can you judge if the average and the equilibrium angle are significantly different?
> - Also estimate the error on the force constant of the quadratic free-energy model of section 4.
>   (Hint: the variance is itself an average, namely of $(\theta-\langle\theta\rangle)^2$.)

## 7. Estimating the initialization phase (or burn-in)

The error estimation in the previous sections only works well when the provided time series is taken from a simulation in equilibrium, i.e. without initialization phase.
One could try to estimate the length of the initialization or burn-in phase by visually inspecting the time dependence in a plot.
However, this is a tedious, subjective and error-prone practice.
In this section, we will employ the scheme proposed in [10.1021/acs.jctc.5b00784](https://doi.org/10.1021/acs.jctc.5b00784) to determine the initialization phase automatically.
This scheme prescribes that the optimal initialization phase maximizes the number of independent samples in the remainder.

The number of independent samples in the remainder is estimated with STACIE, as in the previous section.
Because each STACIE analysis takes a few seconds, the initialization phase is only scanned on a coarse grid of 16 values.
The grid is logarithmic, because the time scale of the initialization is not known in advance
and may range from a few frames to hundreds of frames.
The cell below therefore takes a minute to run.
Because STACIE also provides the uncertainty of each estimate, error bars can be shown in the plot of the number of independent samples.

The example below uses the distance between the nitrogen of ALA18 and the oxygen of SER15, a hydrogen bond in the alpha helix.
During the warming up of the protein, the helix is perturbed and this distance first increases, before it relaxes to its equilibrium value over several picoseconds.

In [ ]:
def scan_initialization(values, timestep, num=16):
    """Scan the number of independent samples in the remainder after an initialization phase.

    Each STACIE analysis takes a few seconds,
    so a coarse logarithmic grid of initialization lengths is used.
    The optimal initialization phase is the one
    that maximizes the number of independent samples in the remainder.

    Returns
    -------
    ninits
        The lengths of the initialization phases (in frames) that were tried.
    num_indeps
        The number of independent samples in the remainder, for each initialization length.
    num_indeps_std
        The uncertainty on the number of independent samples (neglecting
        the uncertainty of the variance of the time series).
    """
    # Zero and a logarithmic grid from 10 frames up to half of the trajectory.
    ninits = np.concatenate(
        [[0], np.unique(np.geomspace(10, len(values) // 2, num - 1).astype(int))]
    )
    num_indeps = np.zeros(len(ninits))
    num_indeps_std = np.zeros(len(ninits))
    for i, ninit in enumerate(ninits):
        remainder = values[ninit:]
        error, _, result = stacie_error(remainder, timestep)
        num_indeps[i] = np.var(remainder) / error**2
        num_indeps_std[i] = num_indeps[i] * result.acint_std / result.acint
    return ninits, num_indeps, num_indeps_std


# Hydrogen bond in the alpha helix: N of ALA18 and O of SER15.
# print(traj.topology.select("protein and ((resSeq 15 and name O) or (resSeq 18 and name N))"))
values = mdtraj.compute_distances(traj, [[262, 224]])[:, 0]
# values = distances[:, 0]
# values = np.degrees(angles[:, 0])
# values = np.degrees(dihedrals[:, 0])
# values = df["Potential Energy (kJ/mole)"].to_numpy()
ninits, num_indeps, num_indeps_std = scan_initialization(values, timestep)
ninit = ninits[np.nanargmax(num_indeps)]

In [ ]:
# Plot the time dependence and the initialization part
plt.close("init")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="init")
ax.plot(df["Time (ps)"][:ninit], values[:ninit], ".", label="Initialization")
ax.plot(df["Time (ps)"][ninit:], values[ninit:], ".", label="Remainder")
ax.axvline(df["Time (ps)"][ninit], color="k")
ax.set_xlabel("Time [ps]")
ax.set_ylabel("Value")
ax.legend(loc=0)

In [ ]:
# Plot of the number of independent samples as function
# of the initialization time.
plt.close("burnin")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="burnin")
ax.errorbar(ninits, num_indeps, num_indeps_std, fmt="o-", color="C2")
ax.axvline(ninit, color="k")
ax.set_xscale("symlog", linthresh=50)
ax.set_xlim(left=0)
ax.set_xlabel("Initialization phase [frames]")
ax.set_ylabel("Number of independent samples")

**Caveat:** the position of the maximum is only a rough estimate of the initialization phase.
The error bars show that the number of independent samples is itself uncertain,
so small differences between neighboring points are not significant.
For slowly decorrelating coordinates, e.g. the dihedral angles or the potential energy,
the time series is too short for a reliable estimate:
the uncertainties are large and the number of independent samples varies erratically with the length of the initialization phase.
In such cases, the maximum can end up at an arbitrary position, and an unrealistically large number of independent samples may even be found.
Always compare the result with the plot of the time dependence, and be cautious when the error bars are large.

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Repeat the above code for a few different time series and answer the following questions:
>
> - Can visual inspection of the initialization phase be too optimistic or pessimistic?
> - What is the accuracy of an error estimate?
>
> An important consideration with the last question is that one has no information of unseen states in an MD simulation.
> For example, a finite MD simulation may be sampling a metastable conformation without relaxing to a more stable one.
> Any averages and error estimates derived from such a run are biased because they are not considering the thermodynamically dominant conformation.